In [ ]:
# Run once → Runtime > Restart
!pip install timm albumentations -q

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, copy, time, shutil, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from tqdm.notebook import tqdm
import cv2
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.optim.swa_utils import AveragedModel, SWALR, update_bn
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.model_selection import train_test_split
from sklearn.metrics import (f1_score, roc_auc_score,
                             confusion_matrix, ConfusionMatrixDisplay)
warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})

class CFG:
    csv_path   = '/content/drive/MyDrive/train1.csv'
    drive_imgs = '/content/drive/MyDrive/images'
    local_imgs = '/content/cxr_images'
    save_path  = '/content/cxr_best.pth'
    drive_save = '/content/drive/MyDrive/cxr_best.pth'

    label_cols   = ['Support Devices', 'Pleural Effusion', 'Lung Opacity']
    num_classes  = 3
    label_smooth = 0.05

    backbone   = 'tf_efficientnetv2_s'   # 24M params, ImageNet-22k pretrained
    img_size   = 288
    batch_size = 48
    grad_accum = 2                       # effective batch = 96
    epochs     = 30
    warmup_ep  = 3
    lr         = 2e-4
    min_lr     = 1e-6
    wd         = 1e-2

    swa_start  = 24                      # SWA over last 6 epochs → +1-2% F1
    swa_lr     = 5e-5

    ema_decay   = 0.9998
    mixup_prob  = 0.35
    mixup_alpha = 0.4
    cutmix_prob = 0.15
    drop_rate   = 0.2
    drop_path   = 0.2

    tta_n    = 5
    thr_min  = 0.35                      # floor blocks trivial all-ones solution
    thr_max  = 0.80

    seed = 42

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.backends.cudnn.benchmark = True

def seed_all(s):
    np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

seed_all(CFG.seed)
gpu_name = torch.cuda.get_device_name(0) if device.type == 'cuda' else 'CPU'
print(f"Device: {device}  |  GPU: {gpu_name}")

Mounted at /content/drive
Device: cuda  |  GPU: Tesla T4


In [ ]:
import os, time, glob, cv2

print("Clearing 135 stale files from SSD...")
os.system(f"rm -f '{CFG.local_imgs}'/*.jpg")

print("Streaming all 16707 images via tar pipe...")
t0 = time.time()
ret = os.system(
    f"tar -C '{CFG.drive_imgs}' -cf - . "
    f"| tar -C '{CFG.local_imgs}' -xf -"
)
elapsed = time.time() - t0
count   = len(glob.glob(os.path.join(CFG.local_imgs, '*.jpg')))

print(f"\nDone in {elapsed:.0f}s — {count}/16707 files on SSD (exit={ret})")

# Verify
print("\nVerifying reads:")
for p in glob.glob(os.path.join(CFG.local_imgs, '*.jpg'))[:4]:
    img = cv2.imread(p)
    ok  = img is not None and img.mean() > 1
    print(f"  {'✓' if ok else '✗ FAIL'} {os.path.basename(p)}"
          + (f"  shape={img.shape} mean={img.mean():.1f}" if img is not None else " → None"))

Clearing 135 stale files from SSD...
Streaming all 16707 images via tar pipe...

Done in 329s — 16707/16707 files on SSD (exit=0)

Verifying reads:
  ✓ 00576_s56134201.jpg  shape=(512, 512, 3) mean=127.5
  ✓ 01490_s52917142.jpg  shape=(512, 512, 3) mean=126.4
  ✓ 11724_s56535553.jpg  shape=(512, 512, 3) mean=123.7
  ✓ 07506_s53960048.jpg  shape=(512, 512, 3) mean=126.0


In [ ]:
# Ensure directory exists
if not os.path.exists(CFG.local_imgs):
    os.makedirs(CFG.local_imgs, exist_ok=True)
    print(f"Created missing directory: {CFG.local_imgs}")

df = pd.read_csv(CFG.csv_path)
df['path'] = df['serial_number'].apply(
    lambda sn: os.path.join(CFG.local_imgs, f"{sn}.jpg")
)

# Filter and check
df = df[df['path'].apply(os.path.exists)].reset_index(drop=True)
print(f"Rows matched: {len(df)}")

if len(df) == 0:
    print("ERROR: No images found. Please ensure images are unzipped to CFG.local_imgs.")
    # Creating a dummy row so the rest of the notebook doesn't crash during debugging
    print("Note: You must fix the image path to proceed with real training.")
else:
    # Stratified split on label-combo key
    df['_k'] = (df[CFG.label_cols] * [4, 2, 1]).sum(axis=1).astype(int)
    train_df, val_df = train_test_split(
        df, test_size=0.15, random_state=CFG.seed,
        stratify=df['_k'], shuffle=True
    )
    train_df = train_df.drop('_k', axis=1).reset_index(drop=True)
    val_df   = val_df.drop('_k', axis=1).reset_index(drop=True)
    print(f"Train {len(train_df)}  Val {len(val_df)}")

    labels_np  = train_df[CFG.label_cols].values.astype(np.float32)
    pos_counts = labels_np.sum(0)
    neg_counts = len(labels_np) - pos_counts
    pos_weight = torch.tensor(neg_counts / (pos_counts + 1e-5)).float().to(device)

    for i, col in enumerate(CFG.label_cols):
        print(f"  {col:20s}  {100*pos_counts[i]/len(labels_np):.1f}%  pw={pos_weight[i]:.2f}")

Created missing directory: /content/cxr_images
Rows matched: 0
ERROR: No images found. Please ensure images are unzipped to CFG.local_imgs.
Note: You must fix the image path to proceed with real training.


In [ ]:
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

# ElasticTransform + GridDistortion removed — each adds 50-80ms/image CPU time
# causing 400+s/epoch. CLAHE + geometric augmentation is sufficient.
train_aug = A.Compose([
    A.CLAHE(clip_limit=4.0, tile_grid_size=(8, 8), p=1.0),
    A.Resize(int(CFG.img_size * 1.10), int(CFG.img_size * 1.10)),
    A.RandomResizedCrop(CFG.img_size, CFG.img_size, scale=(0.80, 1.0)),
    A.HorizontalFlip(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.06, scale_limit=0.12,
                       rotate_limit=12, border_mode=cv2.BORDER_REFLECT, p=0.6),
    A.RandomGamma(gamma_limit=(80, 120), p=0.4),
    A.RandomBrightnessContrast(0.20, 0.20, p=0.5),
    A.OneOf([A.GaussNoise(var_limit=(5, 30)),
             A.GaussianBlur(blur_limit=(3, 5))], p=0.3),
    A.CoarseDropout(max_holes=6, max_height=CFG.img_size // 10,
                    max_width=CFG.img_size // 10, p=0.3),
    A.Normalize(MEAN, STD), ToTensorV2(),
])

val_aug = A.Compose([
    A.CLAHE(clip_limit=4.0, tile_grid_size=(8, 8), p=1.0),
    A.Resize(CFG.img_size, CFG.img_size),
    A.Normalize(MEAN, STD), ToTensorV2(),
])

tta_aug = A.Compose([
    A.CLAHE(clip_limit=4.0, tile_grid_size=(8, 8), p=1.0),
    A.Resize(int(CFG.img_size * 1.06), int(CFG.img_size * 1.06)),
    A.RandomResizedCrop(CFG.img_size, CFG.img_size, scale=(0.88, 1.0)),
    A.HorizontalFlip(p=0.5),
    A.Normalize(MEAN, STD), ToTensorV2(),
])

class CXRDataset(Dataset):
    def __init__(self, df, aug, smooth=0.0):
        self.df, self.aug, self.smooth = df.reset_index(drop=True), aug, smooth

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.loc[idx]
        img = cv2.imread(row['path'], cv2.IMREAD_COLOR)
        img = np.zeros((CFG.img_size, CFG.img_size, 3), dtype=np.uint8) \
              if img is None else cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = self.aug(image=img)['image']
        lbl = row[CFG.label_cols].values.astype(np.float32)
        if self.smooth > 0:
            lbl = lbl * (1 - self.smooth) + 0.5 * self.smooth
        return img, torch.tensor(lbl)

def make_loader(df, aug, shuffle, smooth=0.0):
    return DataLoader(CXRDataset(df, aug, smooth),
                      batch_size=CFG.batch_size, shuffle=shuffle,
                      num_workers=4, prefetch_factor=2,
                      persistent_workers=True, pin_memory=True,
                      drop_last=shuffle)

train_loader = make_loader(train_df, train_aug, True,  CFG.label_smooth)
val_loader   = make_loader(val_df,   val_aug,   False)

t0 = time.time()
_x, _ = next(iter(train_loader))
t1 = time.time() - t0
print(f"Batch time: {t1:.2f}s  |  "
      f"Est. epoch: ~{t1*len(train_loader):.0f}s  |  "
      f"{'OK' if _x.mean().abs() > 0.01 else 'WARNING: black images'}")

ValueError: 1 validation error for InitSchema
size
  Input should be a valid tuple [type=tuple_type, input_value=288, input_type=int]
    For further information visit https://errors.pydantic.dev/2.12/v/tuple_type

In [ ]:
# BCE + pos_weight: 100% negative gradient always (no collapse)
# ASL gamma=3 → negatives get 12.5% gradient → AUC=0.50 trivial collapse
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

def cutmix(imgs, labels, alpha=1.0):
    lam = np.random.beta(alpha, alpha)
    B, C, H, W = imgs.shape
    idx = torch.randperm(B, device=imgs.device)
    cx, cy = np.random.randint(W), np.random.randint(H)
    cw = int(W * (1 - lam) ** .5); ch = int(H * (1 - lam) ** .5)
    x1, x2 = max(0, cx-cw//2), min(W, cx+cw//2)
    y1, y2 = max(0, cy-ch//2), min(H, cy+ch//2)
    imgs[:, :, y1:y2, x1:x2] = imgs[idx, :, y1:y2, x1:x2]
    lam_r = 1 - (x2-x1)*(y2-y1)/(W*H)
    return imgs, lam_r*labels + (1-lam_r)*labels[idx]

class CXRModel(nn.Module):
    """EfficientNetV2-S → LayerNorm → 2-layer MLP head → 3 logits."""
    def __init__(self):
        super().__init__()
        self.backbone = timm.create_model(
            CFG.backbone, pretrained=True, num_classes=0,
            global_pool='avg', drop_rate=CFG.drop_rate,
            drop_path_rate=CFG.drop_path)
        D = self.backbone.num_features
        self.head = nn.Sequential(
            nn.LayerNorm(D), nn.Dropout(0.3),
            nn.Linear(D, 512), nn.SiLU(), nn.Dropout(0.2),
            nn.Linear(512, CFG.num_classes))
        nn.init.xavier_uniform_(self.head[-1].weight)
        nn.init.zeros_(self.head[-1].bias)

    def forward(self, x, return_features=False):
        f = self.backbone(x)
        return (self.head(f), f) if return_features else self.head(f)

class EMA:
    def __init__(self, model, decay=0.9998):
        self.shadow = copy.deepcopy(model).eval().to(device)
        self.decay  = decay
        for p in self.shadow.parameters(): p.requires_grad_(False)

    @torch.no_grad()
    def update(self, model):
        for sp, mp in zip(self.shadow.parameters(), model.parameters()):
            sp.data.mul_(self.decay).add_(mp.data, alpha=1-self.decay)
        for sb, mb in zip(self.shadow.buffers(), model.buffers()):
            sb.copy_(mb)

    def state_dict(self): return self.shadow.state_dict()
    def load_state_dict(self, sd): self.shadow.load_state_dict(sd)

model     = CXRModel().to(device)
ema       = EMA(model, CFG.ema_decay)
swa_model = AveragedModel(model)
head_ids  = set(id(p) for p in model.head.parameters())

optimizer = optim.AdamW([
    {'params': [p for p in model.parameters() if id(p) not in head_ids],
     'lr': CFG.lr/10, 'weight_decay': CFG.wd},
    {'params': list(model.head.parameters()),
     'lr': CFG.lr,    'weight_decay': CFG.wd},
])

def cosine_lr(ep):
    if ep < CFG.warmup_ep: return (ep+1)/CFG.warmup_ep
    prog  = min((ep-CFG.warmup_ep)/max(1, CFG.swa_start-CFG.warmup_ep), 1.0)
    ratio = CFG.min_lr/CFG.lr
    return ratio + 0.5*(1-ratio)*(1+np.cos(np.pi*prog))

scheduler = optim.lr_scheduler.LambdaLR(optimizer, cosine_lr)
swa_sched = SWALR(optimizer, swa_lr=CFG.swa_lr, anneal_epochs=3)
scaler    = torch.amp.GradScaler('cuda')

n_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Backbone : {CFG.backbone}  {n_p/1e6:.1f}M params")
print(f"Loss     : BCE + pos_weight {pos_weight.cpu().numpy().round(3)}")
print(f"Size     : {CFG.img_size}px  batch={CFG.batch_size}  eff={CFG.batch_size*CFG.grad_accum}")

In [ ]:
history = {
    'epoch': [], 'lr': [],
    'train_loss': [], 'train_f1': [],
    'val_f1': [], 'val_auc': [],
    'f1_sd': [], 'f1_pe': [], 'f1_lo': [],
    'auc_sd': [], 'auc_pe': [], 'auc_lo': [],
}

best_f1, best_auc = 0.0, 0.0
best_wts = copy.deepcopy(ema.state_dict())
best_thr = [0.5] * CFG.num_classes
t_start  = time.time()

for epoch in range(CFG.epochs):
    t_ep   = time.time()
    in_swa = epoch >= CFG.swa_start
    lr_now = optimizer.param_groups[-1]['lr']

    print(f"Ep {epoch+1:02d}/{CFG.epochs}  lr={lr_now:.2e}"
          + ("  [SWA]" if in_swa else ""), end="  ")

    # ── train ──────────────────────────────────────────────────
    model.train(); optimizer.zero_grad()
    t_preds, t_true, loss_sum, step = [], [], 0.0, 0

    for imgs, labels in train_loader:
        imgs   = imgs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        rng    = np.random.rand()

        with torch.amp.autocast('cuda'):
            if rng < CFG.mixup_prob:
                lam = np.random.beta(CFG.mixup_alpha, CFG.mixup_alpha)
                idx = torch.randperm(imgs.size(0), device=device)
                out  = model(lam*imgs + (1-lam)*imgs[idx])
                loss = (lam*criterion(out, labels) +
                        (1-lam)*criterion(out, labels[idx])) / CFG.grad_accum
            elif rng < CFG.mixup_prob + CFG.cutmix_prob:
                imgs_c, lbl_c = cutmix(imgs, labels)
                out  = model(imgs_c)
                loss = criterion(out, lbl_c) / CFG.grad_accum
            else:
                out  = model(imgs)
                loss = criterion(out, labels) / CFG.grad_accum

        scaler.scale(loss).backward()
        step += 1
        if step % CFG.grad_accum == 0:
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer); scaler.update()
            optimizer.zero_grad()
            ema.update(model)
            if in_swa: swa_model.update_parameters(model)

        loss_sum += loss.item() * CFG.grad_accum
        t_preds.append((torch.sigmoid(out.detach()) > 0.5).cpu().numpy())
        t_true.append((labels.cpu().numpy() > 0.5).astype(int))

    if not in_swa: scheduler.step()
    else:          swa_sched.step()

    t_loss = loss_sum / len(train_loader)
    t_f1   = f1_score(np.vstack(t_true), np.vstack(t_preds),
                      average='macro', zero_division=0)

    # ── validate (EMA) ─────────────────────────────────────────
    ema.shadow.eval(); v_probs, v_labels = [], []

    with torch.no_grad():
        for imgs, labels in val_loader:
            with torch.amp.autocast('cuda'):
                lg = ema.shadow(imgs.to(device, non_blocking=True))
            v_probs.append(torch.sigmoid(lg).cpu().numpy())
            v_labels.append(labels.numpy())

    v_probs  = np.vstack(v_probs)
    v_labels = np.vstack(v_labels).astype(int)

    # Per-class threshold search — floor 0.35 blocks trivial collapse
    thresholds = []
    for c in range(CFG.num_classes):
        bt, bf = 0.5, 0.0
        for t in np.arange(CFG.thr_min, CFG.thr_max, 0.02):
            cf = f1_score(v_labels[:, c],
                          (v_probs[:, c] > t).astype(int), zero_division=0)
            if cf > bf: bf, bt = cf, t
        thresholds.append(bt)

    v_preds = np.column_stack(
        [(v_probs[:, c] > thresholds[c]).astype(int)
         for c in range(CFG.num_classes)])

    v_f1  = f1_score(v_labels, v_preds, average='macro', zero_division=0)
    v_auc = roc_auc_score(v_labels, v_probs, average='macro')

    per_f1  = [f1_score(v_labels[:,i], v_preds[:,i], zero_division=0)
               for i in range(3)]
    per_auc = [roc_auc_score(v_labels[:,i], v_probs[:,i])
               for i in range(3)]

    elapsed = time.time() - t_ep
    star    = " ★" if v_f1 > best_f1 else ""
    print(f"loss={t_loss:.4f}  F1={v_f1:.4f}  AUC={v_auc:.4f}"
          f"  [{elapsed:.0f}s]{star}")

    # History
    history['epoch'].append(epoch+1)
    history['lr'].append(lr_now)
    history['train_loss'].append(t_loss)
    history['train_f1'].append(t_f1)
    history['val_f1'].append(v_f1)
    history['val_auc'].append(v_auc)
    history['f1_sd'].append(per_f1[0])
    history['f1_pe'].append(per_f1[1])
    history['f1_lo'].append(per_f1[2])
    history['auc_sd'].append(per_auc[0])
    history['auc_pe'].append(per_auc[1])
    history['auc_lo'].append(per_auc[2])

    if v_f1 > best_f1:
        best_f1, best_auc = v_f1, v_auc
        best_wts = copy.deepcopy(ema.state_dict())
        best_thr = thresholds[:]
        torch.save({'state_dict': best_wts, 'thresholds': best_thr,
                    'val_f1': best_f1, 'val_auc': best_auc,
                    'backbone': CFG.backbone, 'label_cols': CFG.label_cols,
                    'img_size': CFG.img_size}, CFG.save_path)

# SWA BN update
update_bn(train_loader, swa_model, device=device)
swa_model.eval(); sw_probs = []
with torch.no_grad():
    for imgs, _ in val_loader:
        with torch.amp.autocast('cuda'):
            lg = swa_model(imgs.to(device))
        sw_probs.append(torch.sigmoid(lg).cpu().numpy())
sw_probs = np.vstack(sw_probs)

sw_thr = []
for c in range(CFG.num_classes):
    bt, bf = 0.5, 0.0
    for t in np.arange(CFG.thr_min, CFG.thr_max, 0.02):
        cf = f1_score(v_labels[:, c], (sw_probs[:, c] > t).astype(int), zero_division=0)
        if cf > bf: bf, bt = cf, t
    sw_thr.append(bt)

sw_preds = np.column_stack([(sw_probs[:,c]>sw_thr[c]).astype(int) for c in range(3)])
sw_f1  = f1_score(v_labels, sw_preds, average='macro', zero_division=0)
sw_auc = roc_auc_score(v_labels, sw_probs, average='macro')

if sw_f1 > best_f1:
    best_f1, best_auc, best_thr = sw_f1, sw_auc, sw_thr
    best_wts = copy.deepcopy(swa_model.module.state_dict())
    torch.save({'state_dict': best_wts, 'thresholds': best_thr,
                'val_f1': best_f1, 'val_auc': best_auc,
                'backbone': CFG.backbone, 'label_cols': CFG.label_cols,
                'img_size': CFG.img_size, 'source': 'SWA'}, CFG.save_path)

total_min = (time.time() - t_start) / 60
print(f"\nTraining done in {total_min:.0f} min")
print(f"Best  F1={best_f1:.4f}  AUC={best_auc:.4f}  "
      f"({'SWA' if sw_f1>best_f1 else 'EMA'} wins)")

In [ ]:
ep  = history['epoch']
col_names = ['Support Devices', 'Pleural Effusion', 'Lung Opacity']
col_short = ['Support Dev.', 'Pleural Eff.', 'Lung Opacity']
colors    = ['#534AB7', '#1D9E75', '#D85A30']   # purple, teal, coral

fig = plt.figure(figsize=(16, 12))
fig.suptitle('CXR Classifier — Training Analysis', fontsize=14, fontweight='500', y=0.98)
gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)

# ── Plot 1: Loss curve ──────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(ep, history['train_loss'], color='#534AB7', linewidth=1.5, label='Train loss')
ax1.axvline(CFG.swa_start, color='#888780', linewidth=0.8, linestyle='--', label='SWA start')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('BCE Loss')
ax1.set_title('Training loss'); ax1.legend(fontsize=8); ax1.grid(alpha=0.2)

# ── Plot 2: Macro F1 ────────────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(ep, history['train_f1'], color='#888780', linewidth=1.2, linestyle='--', label='Train F1')
ax2.plot(ep, history['val_f1'],   color='#534AB7', linewidth=1.8, label='Val F1')
ax2.axvline(CFG.swa_start, color='#888780', linewidth=0.8, linestyle=':')
ax2.axhline(best_f1, color='#534AB7', linewidth=0.6, linestyle=':',
            label=f'Best {best_f1:.3f}')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Macro F1')
ax2.set_title('Macro F1 — train vs val'); ax2.legend(fontsize=8); ax2.grid(alpha=0.2)

# ── Plot 3: Macro AUC ───────────────────────────────────────────
ax3 = fig.add_subplot(gs[0, 2])
ax3.plot(ep, history['val_auc'], color='#1D9E75', linewidth=1.8, label='Val AUC')
ax3.axvline(CFG.swa_start, color='#888780', linewidth=0.8, linestyle=':')
ax3.axhline(best_auc, color='#1D9E75', linewidth=0.6, linestyle=':',
            label=f'Best {best_auc:.3f}')
ax3.set_xlabel('Epoch'); ax3.set_ylabel('Macro AUC')
ax3.set_title('Macro AUC'); ax3.legend(fontsize=8); ax3.grid(alpha=0.2)

# ── Plot 4: Per-class F1 curves ─────────────────────────────────
ax4 = fig.add_subplot(gs[1, :2])
for key, name, c in zip(['f1_sd', 'f1_pe', 'f1_lo'], col_short, colors):
    ax4.plot(ep, history[key], color=c, linewidth=1.5, label=name)
ax4.axvline(CFG.swa_start, color='#888780', linewidth=0.8, linestyle='--', label='SWA start')
ax4.set_xlabel('Epoch'); ax4.set_ylabel('F1')
ax4.set_title('Per-class F1 over training')
ax4.legend(fontsize=8); ax4.grid(alpha=0.2)

# ── Plot 5: Per-class AUC curves ────────────────────────────────
ax5 = fig.add_subplot(gs[1, 2])
for key, name, c in zip(['auc_sd', 'auc_pe', 'auc_lo'], col_short, colors):
    ax5.plot(ep, history[key], color=c, linewidth=1.5, label=name)
ax5.axvline(CFG.swa_start, color='#888780', linewidth=0.8, linestyle='--')
ax5.set_xlabel('Epoch'); ax5.set_ylabel('AUC')
ax5.set_title('Per-class AUC')
ax5.legend(fontsize=8); ax5.grid(alpha=0.2)

# ── Plot 6: Best per-class F1 bar chart ─────────────────────────
ax6 = fig.add_subplot(gs[2, 0])
final_f1s = [history['f1_sd'][-1], history['f1_pe'][-1], history['f1_lo'][-1]]
bars = ax6.bar(col_short, final_f1s, color=colors, width=0.5)
ax6.set_ylim(0, 1.0)
ax6.set_ylabel('F1 Score')
ax6.set_title('Final per-class F1')
for bar, val in zip(bars, final_f1s):
    ax6.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{val:.3f}', ha='center', va='bottom', fontsize=9)
ax6.tick_params(axis='x', labelsize=8); ax6.grid(axis='y', alpha=0.2)

# ── Plot 7: Final per-class AUC bar chart ───────────────────────
ax7 = fig.add_subplot(gs[2, 1])
final_aucs = [history['auc_sd'][-1], history['auc_pe'][-1], history['auc_lo'][-1]]
bars = ax7.bar(col_short, final_aucs, color=colors, width=0.5)
ax7.set_ylim(0, 1.0)
ax7.set_ylabel('AUC')
ax7.set_title('Final per-class AUC')
for bar, val in zip(bars, final_aucs):
    ax7.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{val:.3f}', ha='center', va='bottom', fontsize=9)
ax7.tick_params(axis='x', labelsize=8); ax7.grid(axis='y', alpha=0.2)

# ── Plot 8: Threshold sensitivity (F1 vs threshold) ─────────────
ax8 = fig.add_subplot(gs[2, 2])
thr_range = np.arange(0.10, 0.90, 0.01)
for c, (name, color) in enumerate(zip(col_short, colors)):
    f1s = [f1_score(v_labels[:, c], (v_probs[:, c] > t).astype(int), zero_division=0)
           for t in thr_range]
    ax8.plot(thr_range, f1s, color=color, linewidth=1.2, label=name)
    ax8.axvline(best_thr[c], color=color, linewidth=0.7, linestyle='--')
ax8.set_xlabel('Threshold'); ax8.set_ylabel('F1')
ax8.set_title('F1 vs threshold (dashed = optimal)')
ax8.legend(fontsize=8); ax8.grid(alpha=0.2)

plt.savefig('/content/cxr_analysis.png', dpi=150, bbox_inches='tight',
            facecolor='white')
plt.show()
print("Saved: /content/cxr_analysis.png")

# Summary table
print(f"\n{'─'*52}")
print(f"  {'Metric':<22}  {'Value':>8}")
print(f"{'─'*52}")
print(f"  {'Macro F1 (val)':<22}  {best_f1:>8.4f}")
print(f"  {'Macro AUC (val)':<22}  {best_auc:>8.4f}")
for i, col in enumerate(col_names):
    print(f"  {'F1 '+col[:16]:<22}  {history['f1_sd' if i==0 else 'f1_pe' if i==1 else 'f1_lo'][-1]:>8.4f}")
for i, col in enumerate(col_names):
    print(f"  {'AUC '+col[:15]:<22}  {history['auc_sd' if i==0 else 'auc_pe' if i==1 else 'auc_lo'][-1]:>8.4f}")
print(f"  {'Training time':<22}  {total_min:>7.0f}m")
print(f"{'─'*52}")

In [ ]:
ema.load_state_dict(best_wts)
ema.shadow.eval()

tta_runs = []
for i in range(CFG.tta_n):
    run = []
    tl  = make_loader(val_df, tta_aug, False)
    with torch.no_grad():
        for imgs, _ in tl:
            with torch.amp.autocast('cuda'):
                p = torch.sigmoid(ema.shadow(imgs.to(device, non_blocking=True)))
            run.append(p.cpu().numpy())
    tta_runs.append(np.vstack(run))

# Clean pass double-weighted for stability
clean = []
with torch.no_grad():
    for imgs, _ in val_loader:
        with torch.amp.autocast('cuda'):
            p = torch.sigmoid(ema.shadow(imgs.to(device, non_blocking=True)))
        clean.append(p.cpu().numpy())
cr = np.vstack(clean)
tta_runs += [cr, cr]

tta_probs = np.mean(tta_runs, axis=0)

final_thr = []
for c in range(CFG.num_classes):
    bt, bf = 0.5, 0.0
    for t in np.arange(CFG.thr_min, CFG.thr_max, 0.01):
        cf = f1_score(v_labels[:,c], (tta_probs[:,c]>t).astype(int), zero_division=0)
        if cf > bf: bf, bt = cf, t
    final_thr.append(bt)

final_preds = np.column_stack(
    [(tta_probs[:,c]>final_thr[c]).astype(int) for c in range(CFG.num_classes)])

final_f1  = f1_score(v_labels, final_preds, average='macro', zero_division=0)
final_auc = roc_auc_score(v_labels, tta_probs, average='macro')

# Confusion matrices
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle(f'Confusion matrices (TTA)  —  Macro F1={final_f1:.4f}  AUC={final_auc:.4f}',
             fontsize=12)
for i, (col, ax, color) in enumerate(zip(col_names, axes, colors)):
    cm = confusion_matrix(v_labels[:,i], final_preds[:,i])
    cf = f1_score(v_labels[:,i], final_preds[:,i], zero_division=0)
    ca = roc_auc_score(v_labels[:,i], tta_probs[:,i])
    disp = ConfusionMatrixDisplay(cm, display_labels=['Neg', 'Pos'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(f'{col}\nF1={cf:.3f}  AUC={ca:.3f}  thr={final_thr[i]:.2f}',
                 fontsize=9)

plt.tight_layout()
plt.savefig('/content/cxr_confusion.png', dpi=150, bbox_inches='tight',
            facecolor='white')
plt.show()

print(f"\n{'═'*54}")
print(f"  FINAL TTA RESULTS")
print(f"  Macro F1  = {final_f1:.4f}")
print(f"  Macro AUC = {final_auc:.4f}")
for i, col in enumerate(col_names):
    cf = f1_score(v_labels[:,i], final_preds[:,i], zero_division=0)
    ca = roc_auc_score(v_labels[:,i], tta_probs[:,i])
    print(f"  {col:20s}  F1={cf:.4f}  AUC={ca:.4f}  thr={final_thr[i]:.2f}")
print(f"{'═'*54}")

ckpt = {'state_dict': best_wts, 'thresholds': final_thr,
        'val_f1_tta': final_f1, 'val_auc_tta': final_auc,
        'backbone': CFG.backbone, 'label_cols': CFG.label_cols,
        'img_size': CFG.img_size, 'history': history}
torch.save(ckpt, CFG.save_path)
shutil.copy2(CFG.save_path, CFG.drive_save)
print(f"\nSaved → {CFG.drive_save}")

In [ ]:
def predict(image_path, ckpt_path=CFG.save_path):
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=True)
    m    = CXRModel().to(device)
    m.load_state_dict(ckpt['state_dict'])
    m.eval()
    thr = ckpt['thresholds']

    img = cv2.imread(image_path, cv2.IMREAD_COLOR)
    if img is None: raise FileNotFoundError(image_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    augs = [val_aug] + [tta_aug] * (CFG.tta_n - 1)
    probs_list = []
    with torch.no_grad():
        for aug in augs:
            t = aug(image=img)['image'].unsqueeze(0).to(device)
            with torch.amp.autocast('cuda'):
                lg, feat = m(t, return_features=True)
            probs_list.append(torch.sigmoid(lg).squeeze().cpu().numpy())

    probs = np.mean(probs_list, axis=0)
    preds = (probs > np.array(thr)).astype(int)

    print(f"\n{os.path.basename(image_path)}")
    for i, col in enumerate(CFG.label_cols):
        bar = '█' * int(probs[i]*20) + '░' * (20-int(probs[i]*20))
        verdict = 'YES' if preds[i] else 'NO '
        print(f"  {verdict}  {col:20s}  [{bar}] {probs[i]:.3f}")
    return probs, preds, feat.cpu()

# Example:
# probs, preds, feat = predict('/content/cxr_images/00001_s51136549.jpg')